# 03 — Prepare & Export

Merge the cleaned `irs_edges` + `acs_edges` staging tables into the unified **`migration_flows`** table — one row per **origin × destination × year**, with IRS and ACS measures side by side. Persist `migration_flows` and `nonmigrants` as first-class DuckDB tables, save processed Parquet, and package the sellable export with a codebook.

**`migration_flows` schema (one directed edge per row):**

| group | columns |
|---|---|
| keys | `year`, `origin_fips`, `origin_state`, `origin_state_name`, `origin_type`, `dest_fips`, `dest_state`, `dest_state_name`, `dest_type` |
| IRS | `irs_returns_out/_in`, `irs_individuals_out/_in`, `irs_agi_out/_in` |
| ACS | `acs_migrants`, `acs_moe`, `acs_moe_pct`, `acs_reliable` |

Read "incoming" vs "outgoing" by filtering: rows where `dest_state = X` are moves **into** X; rows where `origin_state = X` are moves **out of** X.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, run_sql, quality_report, save_processed
from src.prepare import (build_migration_flows, build_nonmigrants, package_dataset,
    build_county_edges, build_county_income_class, build_county_migration_flows)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## 1. Merge IRS + ACS into `migration_flows`
Full outer join of `irs_edges` and `acs_edges` on (origin, dest, year), joined to `state_ref` for names/types. Staging tables were created by `02-clean`; if this notebook is run standalone, reload them from interim Parquet first.

In [ ]:
# Ensure staging tables exist (in case 02-clean wasn't run in this session)
have = set(con.execute("SELECT table_name FROM information_schema.tables").df()['table_name'])
interim = Path(cfg['paths']['data_interim'])
for t in ['state_ref', 'irs_edges', 'acs_edges']:
    if t not in have:
        load_to_duckdb(pd.read_parquet(interim / f'{t}.parquet'), t, con)
        print(f'reloaded {t} from interim')

flows = build_migration_flows(con)
print(f'migration_flows: {len(flows):,} rows × {len(flows.columns)} cols')
flows.head()

## 2. Derived reliability columns (ACS)
ACS is a survey, so small flows carry large margins of error. Add:
- `acs_moe_pct` — margin of error as a fraction of the estimate
- `acs_reliable` — `True` when MOE < 50% of the estimate (a common usability cutoff)

Net migration and gross flows are intentionally **not** added here — they are aggregations that belong in `05-analysis`, since they require pairing A→B with B→A.

In [ ]:
load_to_duckdb(flows, '_flows_stage', con)
flows = run_sql("""
    SELECT *,
        CASE WHEN acs_migrants IS NULL OR acs_migrants = 0 THEN NULL
             ELSE ROUND(acs_moe * 1.0 / acs_migrants, 4) END AS acs_moe_pct,
        CASE WHEN acs_migrants IS NULL OR acs_migrants = 0 THEN NULL
             ELSE (acs_moe * 1.0 / acs_migrants) < 0.50 END AS acs_reliable
    FROM _flows_stage
""", con)
con.execute('DROP TABLE IF EXISTS _flows_stage')
print(f'reliable ACS edges: {int(flows.acs_reliable.sum()):,} / {flows.acs_migrants.notna().sum():,} with an estimate')
flows[['year','origin_state','dest_state','acs_migrants','acs_moe','acs_moe_pct','acs_reliable']].head()

## 3. Persist tables to DuckDB
Write `migration_flows` and `nonmigrants` as first-class tables, and register both in `_sources` (using the project's existing schema).

In [ ]:
load_to_duckdb(flows, 'migration_flows', con)

# nonmigrants was built in 02-clean; rebuild here so 03 is self-contained
nonmigrants = build_nonmigrants(con)
load_to_duckdb(nonmigrants, 'nonmigrants', con)

from datetime import date
today = date.today().isoformat()
def register(table, source_name, source_url, description, n):
    con.execute('INSERT OR REPLACE INTO _sources (table_name, source_name, source_url, description, retrieved_date, row_count) VALUES (?, ?, ?, ?, ?, ?)',
                [table, source_name, source_url, description, today, int(n)])

register('migration_flows', 'IRS SOI + Census ACS (merged)', '',
         'Unified directed-edge migration table: one row per origin x destination x year, IRS (out/in) and ACS (migrants/MOE) side by side. True state-to-state edges only.', len(flows))
register('nonmigrants', 'IRS SOI Migration Data', 'https://www.irs.gov/statistics/soi-tax-stats-migration-data',
         'IRS rows where origin_fips==dest_fips (filed in same state both years). One row per state x year.', len(nonmigrants))
print('persisted migration_flows and nonmigrants to DuckDB')

## 4. Quality report

In [ ]:
_ = quality_report(flows, 'migration_flows', con,
                   required_columns=['year','origin_fips','dest_fips','origin_state','dest_state'],
                   max_null_pct=0.60)
print()
# grain check: (origin, dest, year) must be unique
dupes = run_sql("""
    SELECT origin_fips, dest_fips, year, COUNT(*) c
    FROM migration_flows GROUP BY 1,2,3 HAVING COUNT(*) > 1
""", con)
print(f'duplicate (origin,dest,year) keys: {len(dupes)}  (should be 0)')

## 5. Save processed Parquet

In [ ]:
save_processed(flows,       cfg, 'migration_flows.parquet')
save_processed(nonmigrants, cfg, 'nonmigrants.parquet')

## 7. County-to-county flows with income-class tags
Join the county edges to each county's income class (above/below national median AGI-per-return) on both endpoints, producing `county_migration_flows` — one row per origin county × dest county × year with a `flow_class` (e.g. `above_to_below`).

If 02-clean wasn't run this session, the staging tables are rebuilt from `county_raw` first.

In [ ]:
have = set(con.execute('SELECT table_name FROM information_schema.tables').df()['table_name'])
if 'county_edges' not in have:
    load_to_duckdb(build_county_edges(con), 'county_edges', con)
if 'county_income_class' not in have:
    from src.prepare import build_county_nonmigrants
    load_to_duckdb(build_county_nonmigrants(con), 'county_nonmigrants', con)
    load_to_duckdb(build_county_income_class(con), 'county_income_class', con)

county_flows = build_county_migration_flows(con)
load_to_duckdb(county_flows, 'county_migration_flows', con)
print(f'county_migration_flows: {len(county_flows):,} rows')
print(con.execute('SELECT flow_class, COUNT(*) edges, SUM(irs_individuals_out) people FROM county_migration_flows GROUP BY flow_class ORDER BY edges DESC').df().to_string(index=False))

from datetime import date
today = date.today().isoformat()
IRS_URL = 'https://www.irs.gov/statistics/soi-tax-stats-migration-data'
register = lambda t,sn,su,d,n: con.execute('INSERT OR REPLACE INTO _sources (table_name, source_name, source_url, description, retrieved_date, row_count) VALUES (?,?,?,?,?,?)',[t,sn,su,d,today,int(n)])
register('county_migration_flows', 'IRS SOI Migration Data (derived)', IRS_URL,
         'County-to-county directed flows, 2023, with origin/dest income class (above/below national median AGI-per-return) and combined flow_class.', len(county_flows))

save_processed(county_flows, cfg, 'county_migration_flows.parquet')

### Package county export (CSV + Excel + Parquet + codebook)

In [ ]:
county_codebook = {
    'year': 'Migration year = second year of the IRS pair (2023 = the 2022–2023 move).',
    'origin_fips': '5-digit county GEOID people moved FROM (state FIPS + county FIPS).',
    'origin_name': 'Origin county name.',
    'origin_agi_per_return': 'Origin county income proxy: non-migrant AGI per return, in dollars.',
    'origin_income_class': "'above_median' or 'below_median' vs the national median county AGI-per-return.",
    'dest_fips': '5-digit county GEOID people moved TO.',
    'dest_name': 'Destination county name.',
    'dest_agi_per_return': 'Destination county income proxy: non-migrant AGI per return, in dollars.',
    'dest_income_class': "'above_median' or 'below_median' for the destination county.",
    'flow_class': 'Combined origin→dest income class: above_to_above / above_to_below / below_to_above / below_to_below.',
    'irs_returns_out': 'IRS returns (~households) for this county move, from the origin outflow file.',
    'irs_returns_in': 'IRS returns from the destination inflow file.',
    'irs_individuals_out': 'IRS individuals (~people), origin outflow file.',
    'irs_individuals_in': 'IRS individuals, destination inflow file.',
    'irs_agi_out': 'AGI for this move (THOUSANDS of dollars), origin outflow file.',
    'irs_agi_in': 'AGI (THOUSANDS of dollars), destination inflow file.',
}
county_notes = '''
Source: IRS SOI county-to-county migration (per-state Excel files) — https://www.irs.gov/statistics/soi-tax-stats-migration-data (public domain)

Grain: one row per origin county x destination county x year (directed edge). True county-to-county moves only;
summary rows, foreign, and non-migrant self-edges excluded (non-migrants used only to derive the income proxy).

Income class: each county is flagged above/below the NATIONAL MEDIAN of county AGI-per-return, where a county's
AGI-per-return is its non-migrant total AGI / non-migrant returns (a proxy for resident income). The median is
computed across all ~3,140 counties, one vote per county. This is a county-level income proxy, NOT the income of
the people in each specific flow (IRS does not publish migration by income bracket).
'''
written_county = package_dataset(county_flows, cfg, name='county_migration_flows_v1',
                                 codebook=county_codebook, notes=county_notes)
written_county

## 6. Package export (CSV + Excel + Parquet + codebook)

In [ ]:
codebook = {
    'year': 'Migration year = the later of the two years compared (IRS 2022–23 pair and ACS 2023 survey both map to 2023).',
    'origin_fips': 'FIPS code of the origin (state people moved FROM). Non-state codes: 72=Puerto Rico, 57=Foreign Country; U.S. Island Areas uses sentinel US-ISL.',
    'origin_state': 'Two-letter abbreviation of the origin.',
    'origin_state_name': 'Full name of the origin.',
    'origin_type': 'Origin location type: state, dc, territory (PR / U.S. Island Areas), or foreign.',
    'dest_fips': 'FIPS code of the destination (state people moved TO). Same coding as origin_fips.',
    'dest_state': 'Two-letter abbreviation of the destination.',
    'dest_state_name': 'Full name of the destination.',
    'dest_type': 'Destination location type: state, dc, territory, or foreign.',
    'irs_returns_out': 'IRS: number of tax returns (~households) for this origin→dest move, as reported in the ORIGIN state outflow file. Raw counts (not thousands).',
    'irs_returns_in': 'IRS: number of tax returns (~households) for this origin→dest move, as reported in the DESTINATION state inflow file. Usually equals irs_returns_out; differs slightly in later series.',
    'irs_individuals_out': 'IRS: number of personal exemptions (~people) for this move, from the origin outflow file.',
    'irs_individuals_in': 'IRS: number of personal exemptions (~people) for this move, from the destination inflow file.',
    'irs_agi_out': 'IRS: total adjusted gross income for this move (in THOUSANDS of dollars), from the origin outflow file.',
    'irs_agi_in': 'IRS: total adjusted gross income for this move (in THOUSANDS of dollars), from the destination inflow file.',
    'acs_migrants': 'Census ACS: estimated number of people who made this origin→dest move (survey estimate).',
    'acs_moe': 'Census ACS: margin of error (± people) on acs_migrants at 90% confidence.',
    'acs_moe_pct': 'acs_moe as a fraction of acs_migrants. Higher = less reliable.',
    'acs_reliable': 'True when acs_moe is less than 50% of acs_migrants (a usability cutoff for small survey flows).',
}

notes = '''
Sources:
  - IRS Statistics of Income (SOI) State-to-State Migration Data — https://www.irs.gov/statistics/soi-tax-stats-migration-data (public domain)
  - U.S. Census Bureau, American Community Survey (ACS) State-to-State Migration Flows — https://www.census.gov/data/tables/time-series/demo/geographic-mobility/state-to-state-migration.html (public domain)

Grain: one row per origin x destination x year (directed edge). True state-to-state moves only;
IRS summary rows (FIPS 96/97/98), overseas (59), and non-migrant (stayed-put) rows are excluded
(non-migrants live in a separate nonmigrants table).

Coverage: IRS 2012–2023 (year = second year of the pair); ACS 2011–2019, 2021–2024 (no 2020, ACS 1-year
suspended for COVID). Rows carry nulls where only one source covers that year.

Caveats:
  - IRS counts tax filers, not total population; ACS counts all residents 1 year and older.
  - IRS AGI is in THOUSANDS of dollars.
  - ACS flows are survey estimates; use acs_reliable / acs_moe_pct before headlining small numbers.
  - 2022 ACS Connecticut data has a known processing error (corrected in 2023).
'''

written = package_dataset(flows, cfg, name='state_migration_flows_v1',
                          codebook=codebook, notes=notes)
written

---
**Next:** `05-analysis.ipynb` for net-migration leaders/losers, income-weighted flows, and tax correlations; `04-viz.ipynb` for exploratory charts.

---
## Cleanup
Close the DuckDB connection so the write lock is released for other tools (DBCode, other notebooks).

In [ ]:
con.close()
print('connection closed')